# Methods for MixModResults Objects (Python)

**Author:** Dimitris Rizopoulos

This notebook mirrors the R vignette *Methods for MixMod Objects* and demonstrates
the Python API for models fitted by `MixedModel` in **glmmadaptive**.

## Implementation status

| R method | Python equivalent | Status |
|---|---|---|
| `print()` / `summary()` | `repr(fm)` / `fm.summary()` | **Implemented** |
| `confint()` fixed effects | `fm.confint()` | **Implemented** |
| `confint(parm="var-cov")` | — | Not yet implemented |
| `vcov()` | `fm.vcov()` | **Implemented** |
| `vcov(sandwich=TRUE)` | — | Not yet implemented |
| `fixef()` | `fm.fixef()` | **Implemented** |
| `ranef()` | `fm.ranef()` | **Implemented** |
| `coef()` (subject-specific) | see note | Partial |
| `marginal_coefs()` | — | Not yet implemented |
| `fitted()` | `fm.fitted()` | **Implemented** |
| `fitted(type="marginal")` | — | Not yet implemented |
| `residuals()` | `fm.residuals()` | **Implemented** |
| `effectPlotData()` | manual via `predict()` | **Implemented** |
| `effects` package | — | No Python equivalent (see §7) |
| `anova()` | `MixModResults.anova()` | **Implemented** |
| `predict()` mean/subject | `fm.predict()` | **Implemented** |
| `predict(type="marginal")` | — | Not yet implemented |
| `predict(newdata2=...)` | `fm.predict_dynamic()` | **Implemented** |
| `simulate()` | — | Not yet implemented |

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.special import expit

## 1  Data simulation and model fitting

In [ ]:
rng = np.random.default_rng(1234)
n, K, t_max = 100, 8, 15.0
times = np.concatenate(
    [np.concatenate([[0.0], np.sort(rng.uniform(0, t_max, K - 1))])
     for _ in range(n)]
)
ids = np.repeat(np.arange(n), K)
sex = np.repeat(np.tile(["male", "female"], n // 2), K)
DF  = pd.DataFrame({"id": ids, "time": times, "sex": sex})

betas_true = np.array([-2.13, -0.25, 0.24, -0.05])
D11, D22   = 0.48, 0.10
b0 = rng.normal(0, np.sqrt(D11), n)
b1 = rng.normal(0, np.sqrt(D22), n)
sex_f = (DF["sex"] == "female").astype(float).values
X_sim = np.column_stack([np.ones(n*K), sex_f, DF["time"].values, sex_f*DF["time"].values])
Z_sim = np.column_stack([np.ones(n*K), DF["time"].values])
eta   = X_sim @ betas_true + (Z_sim * np.column_stack([b0[ids], b1[ids]])).sum(1)
DF["y"] = rng.binomial(1, expit(eta))
print(DF.head(10))
print(f"\nEvent rate: {DF.y.mean():.3f}")

In [ ]:
from glmmadaptive import MixedModel
from glmmadaptive.families import Binomial
from glmmadaptive.results import MixModResults

fm = MixedModel(
    fixed="y ~ sex * time", random="~ time | id",
    data=DF, family=Binomial(),
    control={"iter_em": 50, "verbose": False},
).fit()
print(repr(fm))

## 2  Summary, confidence intervals, and covariance matrix

In [ ]:
print(fm.summary())

In [ ]:
ci = fm.confint(level=0.95)
print("95% CIs (log-odds scale):")
print(ci)
print("\n95% CIs (odds-ratio scale):")
print(np.exp(ci))

In [ ]:
print("90% CIs:")
print(fm.confint(level=0.90))

> **Note:** `confint(parm="var-cov")` (variance components) and `vcov(sandwich=TRUE)` (robust SEs) are not yet implemented.

In [ ]:
print("Vcov (fixed effects):")
print(np.round(fm.vcov(), 6))

## 3  Fixed and random effects

In [ ]:
print("Fixed effects:")
print(fm.fixef())

In [ ]:
print("Random effects (first 6 subjects):")
print(fm.ranef().head(6))

> **Note:** In R, `coef(fm)` returns subject-specific coefficients (fixed + random). In Python, `fm.coef()` is an alias for `fm.fixef()`. Assemble manually:

In [ ]:
re = fm.ranef()
fe = fm.fixef().values
coef_subj = pd.DataFrame({
    "(Intercept)": fe[0] + re.iloc[:, 0],
    "time":        fe[2] + re.iloc[:, 1],
}, index=re.index)
print(coef_subj.head(6))

## 4  Marginalized coefficients

> **Not yet implemented.** In R, `marginal_coefs(fm)` computes population-averaged log-odds ratios via Monte Carlo (Hedeker et al., 2018). A Python implementation is planned.

## 5  Fitted values and residuals

In [ ]:
fit_ms = fm.fitted(type_="mean_subject")
fit_ss = fm.fitted(type_="subject_specific")
print("Mean-subject (first 8):    ", np.round(fit_ms[:8], 4))
print("Subject-specific (first 8):", np.round(fit_ss[:8], 4))

In [ ]:
resid_ms = fm.residuals(type_="mean_subject")
resid_ss = fm.residuals(type_="subject_specific")
print(f"Mean residual (mean-subject):     {resid_ms.mean():.4f}")
print(f"Mean residual (subject-specific): {resid_ss.mean():.4f}")

## 6  Effect plots

The Python equivalent of `effectPlotData()` uses `predict()` on a covariate grid with delta-method confidence intervals.

In [ ]:
import itertools, patsy

times_grid = np.linspace(DF["time"].min(), DF["time"].max(), 25)
nDF = pd.DataFrame(list(itertools.product(times_grid, ["male","female"])),
                   columns=["time","sex"])
nDF["id"] = "new"

pred_ms = fm.predict(newdata=nDF, type_="mean_subject")

_, X_new = patsy.dmatrices(fm.fixed_formula, nDF, return_type="matrix")
X_new    = np.asarray(X_new)
se_eta   = np.sqrt(np.einsum("ij,jk,ik->i", X_new, fm.vcov(), X_new))
eta_new  = X_new @ fm.fixef().values

nDF["pred"]   = pred_ms
nDF["low"]    = expit(eta_new - 1.96 * se_eta)
nDF["upp"]    = expit(eta_new + 1.96 * se_eta)
nDF["eta"]    = eta_new
nDF["eta_lo"] = eta_new - 1.96 * se_eta
nDF["eta_hi"] = eta_new + 1.96 * se_eta

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, grp in zip(axes, ["male", "female"]):
    sub = nDF[nDF["sex"] == grp]
    ax.plot(sub["time"], sub["pred"], color="red", lw=2, label="Predicted")
    ax.fill_between(sub["time"], sub["low"], sub["upp"],
                    color="grey", alpha=0.3, label="95% CI")
    ax.set_title(grp); ax.set_xlabel("Follow-up time")
    ax.set_ylabel("Subject-specific probability"); ax.set_ylim(0, 1); ax.legend(fontsize=8)
plt.suptitle("Effect plot — probability scale"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, grp in zip(axes, ["male", "female"]):
    sub = nDF[nDF["sex"] == grp]
    ax.plot(sub["time"], sub["eta"], color="red", lw=2, label="Log-odds")
    ax.fill_between(sub["time"], sub["eta_lo"], sub["eta_hi"],
                    color="grey", alpha=0.3, label="95% CI")
    ax.set_title(grp); ax.set_xlabel("Follow-up time")
    ax.set_ylabel("Log-odds"); ax.legend(fontsize=8)
plt.suptitle("Effect plot — log-odds scale"); plt.tight_layout(); plt.show()

## 7  Effect plots using the effects package

> **R only.** The R **effects** package (`predictorEffect("time", fm)`) has no direct Python equivalent.
>
> Closest Python alternatives:
> - [`marginaleffects`](https://marginaleffects.com/) — average marginal effects; GLMM support pending.
> - `statsmodels` — marginal effects for GLMs, not GLMMs.
> - The manual grid approach in §6 covers most practical use cases.

## 8  Comparing two models

In [ ]:
gm = MixedModel(
    fixed="y ~ sex * time", random="~ 1 | id",
    data=DF, family=Binomial(),
    control={"iter_em": 50, "verbose": False},
).fit()
print(MixModResults.anova(gm, fm))

## 9  Predictions for new subjects

In [ ]:
pred_DF = DF[DF["id"] == 0].head(4).copy()
pred_DF["id"] = "N1"
print(pred_DF[["id","time","sex","y"]])

In [ ]:
preds_ms = fm.predict(newdata=pred_DF, type_="mean_subject")
print("Mean-subject predictions:")
print(pd.DataFrame({"time": pred_DF["time"].values, "pred": np.round(preds_ms, 4)}))

In [ ]:
preds_ss = fm.predict(newdata=pred_DF, type_="subject_specific")
print("Subject-specific predictions:")
print(pd.DataFrame({"time": pred_DF["time"].values, "pred": np.round(preds_ss, 4)}))

> **Note:** `predict(type="marginal")` is not yet implemented.

### Dynamic predictions for future time points

In [ ]:
future_times = pred_DF[["id","time","sex"]].iloc[:3].copy()
future_times["time"] = [3.0, 4.0, 10.0]
print(future_times)

In [ ]:
dyn = fm.predict_dynamic(newdata=pred_DF, newdata2=future_times)
print(pd.DataFrame({"time": future_times["time"].values,
                    "pred": np.round(dyn["predictions"], 4)}))

## 10  Simulate

> **Not yet implemented.** In R, `simulate(fm, nsim=2, seed=123)` draws replicate responses:
> 1. Draw $b_i \sim N(0, \hat{D})$.
> 2. Compute $\eta_i = X_i\hat{\beta} + Z_i b_i$.
> 3. Draw $y_i \sim F(\text{linkinv}(\eta_i))$.
>
> A `simulate()` method is planned. Use `fm.fixef()`, `fm.D`, and `fm.family.linkinv()` directly in the meantime.